# PIP client examples

## examples/01_local_quickstart.py

In [1]:
from featuremesh import BatchClient

# Create a client (local mode + DuckDB is the default -- no token, no server needed)
client = BatchClient()

# Translate and execute a FeatureQL query
result = client.query("""
    SELECT
        F1 := 1,
        F2 := 2,
        F3 := F1 + F2;
""")

print(result.dataframe)
#    F1  F2  F3
# 0   1   2   3

# Translate only (get the generated SQL without executing)
translate_result = client.translate("""
    WITH
        ID := INPUT(BIGINT)
    SELECT
        ID,
        DOUBLED := ID * 2
    FOR
        ID := BIND_VALUES(ARRAY[1, 2, 3]);
""")

print(translate_result.sql)
print(translate_result.success)

   F1  F2  F3
0   1   2   3
/* START_OF_QUERY reference: default */
WITH
CTE_2EDS2F1N_0 AS (
            (
                SELECT UNNEST(
            CAST(ARRAY[1, 2, 3] AS BIGINT[])
        ) AS ID
        )
), 
CTE_2EDS2F1N_1 AS (
SELECT
	CAST(ID AS BIGINT) AS ID
FROM CTE_2EDS2F1N_0
), 
CTE_2EDS2F1N_2 AS (
SELECT
	CAST(ID * 2 AS BIGINT) AS DOUBLED, 
	ID
FROM CTE_2EDS2F1N_1
)
/*
reference = default [final layer=3]
CteTypeEnum.FINAL
*/
SELECT
    ID, DOUBLED
FROM CTE_2EDS2F1N_2
/* END_OF_QUERY reference: default */

True


## examples/02_local_persistence.py

In [2]:
from featuremesh import BatchClient

# Local mode persists features in a SQLite file (default: ./featuremesh.db)
client = BatchClient()

# Create features in a namespace
client.query("""
    CREATE OR REPLACE FEATURES IN fm.demo AS
    SELECT
        CUSTOMERS := ENTITY(),
        CUSTOMER_ID := INPUT(BIGINT#CUSTOMERS),
        GREETING := 'Hello, customer #' || UNSAFE_CAST(CUSTOMER_ID AS VARCHAR);
""")

# Explore what was persisted
result = client.query("SHOW FEATURES;")
print("Persisted features:")
print(result.dataframe)

# Query the persisted features by referencing the namespace
result = client.query("""
    SELECT
        CUSTOMER_ID,
        GREETING
    FROM fm.demo
    FOR
        CUSTOMER_ID := BIND_VALUES(ARRAY[1, 2, 3]);
""")
print(result.dataframe)

# Clean up
client.query("DROP FEATURES IF EXISTS fm.demo.GREETING, fm.demo.CUSTOMER_ID, fm.demo.CUSTOMERS;")

Persisted features:
                  NAME                  DATATYPE   FUNCTION  \
0     FM.DEMO.GREETING                   VARCHAR  CONCAT_FN   
1  FM.DEMO.CUSTOMER_ID  BIGINT#FM.DEMO.CUSTOMERS      INPUT   
2    FM.DEMO.CUSTOMERS                    ENTITY     ENTITY   

                    INPUTS                                            FORMULA  \
0  ["FM.DEMO.CUSTOMER_ID"]  'Hello, customer #' || UNSAFE_CAST(FM.DEMO.CUS...   
1                       []                    INPUT(BIGINT#FM.DEMO.CUSTOMERS)   
2                       []                                           ENTITY()   

  RESTRICTIONS STATUS                                       DEPENDENCIES  \
0           []    DEV  {"FM.DEMO.CUSTOMER_ID": 1, "UNNAMED_FEATURE_KC...   
1           []    DEV                         {"FM.DEMO.CUSTOMER_ID": 1}   
2           []    DEV                           {"FM.DEMO.CUSTOMERS": 1}   

               CREATED_AT  CREATED_BY              UPDATED_AT  UPDATED_BY  
0 2026-04-12 13:01:27

QueryResult(featureql='DROP FEATURES IF EXISTS fm.demo.GREETING, fm.demo.CUSTOMER_ID, fm.demo.CUSTOMERS;', sql="\n                    SELECT feature_name, status, message\n                    \n                    FROM (VALUES ('FM.DEMO.GREETING', 'DELETED', 'Feature successfully deleted'), ('FM.DEMO.CUSTOMER_ID', 'DELETED', 'Feature successfully deleted'), ('FM.DEMO.CUSTOMERS', 'DELETED', 'Feature successfully deleted')) AS t(feature_name, status, message)\n                    \n                ", dataframe=          feature_name   status                       message
0     FM.DEMO.GREETING  DELETED  Feature successfully deleted
1  FM.DEMO.CUSTOMER_ID  DELETED  Feature successfully deleted
2    FM.DEMO.CUSTOMERS  DELETED  Feature successfully deleted, slt='# schema: VARCHAR|VARCHAR|VARCHAR\nquery feature_name:V,status:V,message:V\nDROP FEATURES IF EXISTS fm.demo.GREETING, fm.demo.CUSTOMER_ID, fm.demo.CUSTOMERS;\n----\nFM.DEMO.GREETING\tDELETED\tFeature successfully deleted\nFM.DEMO.CU

## examples/03_managed_batch.py

In [3]:
import featuremesh as fm

# START - DO NOT SHIP THIS, JUST FOR LOCAL TESTING
from libs.helpers.utils import get_featuremesh_config
fm_config = get_featuremesh_config()
fm.set_default('managed.host', fm_config['managed.host'])
fm.set_default('access.host', fm_config['access.host'])
# END - DO NOT SHIP THIS, JUST FOR LOCAL TESTING

# Switch to managed mode (uses FeatureMesh's remote registry API)
fm.set_default("registry", fm.Registry.MANAGED)

# Get your access token from https://console.featuremesh.com/login
__YOUR_IDENTITY_TOKEN__ = fm_config['identity_token']
__YOUR_ACCESS_TOKEN__ = fm.generate_access_token(identity_token=__YOUR_IDENTITY_TOKEN__, project='default')


# Provide a SQL executor for your database
def query_duckdb(sql: str):
    import duckdb
    return duckdb.sql(sql).df()


client = fm.BatchClient(
    access_token=__YOUR_ACCESS_TOKEN__,
    backend=fm.Backend.DUCKDB,
    sql_executor=query_duckdb,
)

result = client.query("""
    WITH
        ID := INPUT(BIGINT)
    SELECT
        ID,
        DOUBLED := ID * 2
    FOR
        ID := BIND_VALUES(ARRAY[1, 2, 3]);
""")

print(result.dataframe)

# Reset to local mode when done
fm.set_default("registry", fm.Registry.LOCAL)

   ID  DOUBLED
0   1        2
1   2        4
2   3        6


## examples/04_jupyter_magic.py

In [4]:
# Step 1: Load the extension
%reload_ext featuremesh

# Step 2: Create and register a default client
from featuremesh import BatchClient, set_default

client = BatchClient()

# Step 3: Use the %%featureql cell magic in notebook cells:
#
# %%featureql
# SELECT F1 := 1, F2 := 2, F3 := F1 + F2;
#
# Result:
# | F1 | F2 | F3 |
# |----|----|----|
# |  1 |  2 |  3 |

# Available magic options:
#   %%featureql --show-sql          Print the generated SQL alongside results
#   %%featureql --debug             Enable debug mode
#   %%featureql --hide-dataframe    Suppress DataFrame output
#   %%featureql --show-slt          Print SLT test format
#   %%featureql --hook results      Store result dict in a notebook variable
#   %%featureql --client my_client  Use a specific client variable

In [5]:
%%featureql
SELECT F1 := 1, F2 := 2, F3 := F1 + F2;

,F1,F2,F3
0,1,2,3


In [6]:
fm.help()

FeatureMesh -- FeatureQL client

Search documentation, code samples, signatures, and tests on a client:

    from featuremesh import BatchClient
    client = BatchClient()
    client.help('zip', 'date_diff').display()

Optional extras: pip install featuremesh[trino]  or  [bigquery]  or  [all]

Full guide: https://featuremesh.com/docs/getting_started/python_library



In [7]:
%%featureql
SHOW DOCS (INCLUDE (CONTENT)) WHERE CONTENT LIKE '%RELATED%';

,NAME,CATEGORY,DISPLAY_ORDER,TITLE,CONTENT
0,featureql/syntax_reference/all_functions/+page.md,SYNTAX_REFERENCE,[],All Functions,---\ntitle: All Functions\ntitle_sidebar: All ...
1,featureql/syntax_reference/all_functions/core/...,SYNTAX_REFERENCE,[],CORE Functions,---\ntitle: CORE Functions\ntitle_sidebar: COR...
2,1-concepts/2-design_philosophy.md,DOC_PAGE,"[1, 2]",Design philosophy,---\ntitle_page: Design philosophy\ntitle_side...
3,1-concepts/3-in_your_organisation.md,DOC_PAGE,"[1, 3]",FeatureQL in your organization,---\ntitle_page: FeatureQL in your organizatio...
4,1-concepts/4-batch_analytics.md,DOC_PAGE,"[1, 4]",Batch analytics,---\ntitle_page: Batch analytics\ntitle_sideba...
5,featureql/syntax_reference/all_functions/core/...,SYNTAX_REFERENCE,"[2, 0]",RELATED,---\ntitle: RELATED\n---\n\n[All functions](/d...
6,2-getting_started/2-python_library.md,DOC_PAGE,"[2, 2]",Python library,---\ntitle_page: Python library\ntitle_sidebar...
7,2-getting_started/3-for_the_impatient.md,DOC_PAGE,"[2, 3]",FeatureQL for the Impatient,---\ntitle_page: FeatureQL for the Impatient\n...
8,3-featureql/1-foundations/2-types.md,DOC_PAGE,"[3, 1, 2]",Types,---\ntitle_page: Types\ntitle_sidebar: Types\n...
9,3-featureql/2-structural_operations/5-related.md,DOC_PAGE,"[3, 2, 5]",Join on single key with RELATED(),---\ntitle_page: Join on single key with RELAT...
